# Unidad 2 — ADN, ARN y Proteínas en Python
### Análisis de Datos en Biotecnología Aplicada con Python · Escuela Global
**Docente:** Carlos Cárdenas Fernández

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/unimauro/analisis-datos-biotecnologia-python/blob/main/notebooks/U2_practica_adn_arn_proteinas.ipynb)

En esta práctica vamos a representar y analizar información biológica usando solo Python.
Sigue las celdas en orden y ejecuta cada una con **Shift + Enter**.

**Lo que harás:**
1. Tratar el ADN como datos (texto)
2. Medir composición y contenido GC
3. Calcular la cadena complementaria y reversa-complementaria
4. Transcribir ADN → ARN
5. Traducir ARN → Proteína
6. Usar **Biopython** (la forma profesional)
7. Caso en salud: detectar una mutación
8. **Bonus: visualizar el ADN en 3D** 🧬
9. Retos para resolver

## 1. El ADN como un string
Una secuencia de ADN es, para la computadora, **solo texto**: las letras A, C, G y T.

In [ ]:
adn = "ATGCTAGCTAGCTAGCATGC"

print("Secuencia:", adn)
print("Longitud :", len(adn))
print("Primera base:", adn[0])
print("Cantidad de A:", adn.count("A"))
print("Cantidad de cada base:")
for base in "ACGT":
    print(f"   {base}: {adn.count(base)}")

## 2. Contenido GC
El **%GC** = proporción de bases G y C. Indica estabilidad térmica del ADN y sirve para comparar genes y organismos.

In [ ]:
def contenido_gc(seq):
    seq = seq.upper()
    g = seq.count("G")
    c = seq.count("C")
    return (g + c) / len(seq) * 100

print(f"%GC de '{adn}': {contenido_gc(adn):.1f}%")
print(f"%GC de 'ATGCGCGC'   : {contenido_gc('ATGCGCGC'):.1f}%")

## 3. Complemento y reversa-complementaria
Las bases se emparejan: **A–T** y **G–C**. La hebra reversa-complementaria es la "otra cara" del ADN.

In [ ]:
comp = {"A": "T", "T": "A", "G": "C", "C": "G"}

def complementaria(seq):
    return "".join(comp[b] for b in seq.upper())

def reversa_complementaria(seq):
    return "".join(comp[b] for b in reversed(seq.upper()))

print("Original        :", adn)
print("Complementaria  :", complementaria(adn))
print("Reversa-comp.   :", reversa_complementaria(adn))

## 4. Transcripción: ADN → ARN
Regla simple: la **T** (timina) se convierte en **U** (uracilo).

In [ ]:
def transcribir(adn):
    return adn.upper().replace("T", "U")

arn = transcribir(adn)
print("ADN:", adn)
print("ARN:", arn)

## 5. Traducción: ARN → Proteína
El ARN se lee de **3 en 3** (codones). Cada codón codifica un aminoácido. Aquí usamos una tabla reducida; más abajo usaremos la tabla completa de Biopython.

In [ ]:
codigo = {
    "AUG": "M", "UUU": "F", "UUC": "F", "GGC": "G", "GCU": "A",
    "CUA": "L", "AGC": "S", "UAA": "*", "UAG": "*", "UGA": "*",
}

def traducir(arn):
    prot = ""
    for i in range(0, len(arn) - 2, 3):
        codon = arn[i:i+3]
        aa = codigo.get(codon, "?")
        if aa == "*":
            break
        prot += aa
    return prot

print("ARN     :", arn)
print("Proteína:", traducir(arn))
print("(El '?' aparece si falta un codón en nuestra tabla reducida — por eso usamos Biopython.)")

## 6. Biopython: la forma profesional
No reinventemos la rueda. **Biopython** trae el objeto `Seq` con métodos listos y la tabla de codones estándar.

> En Colab, la siguiente celda instala Biopython (tarda unos segundos).

In [ ]:
!pip install -q biopython

In [ ]:
from Bio.Seq import Seq

gen = Seq("ATGCTAGCTAGCTAGTAA")
print("Secuencia      :", gen)
print("Complementaria :", gen.complement())
print("Reversa-comp.  :", gen.reverse_complement())
print("Transcripción  :", gen.transcribe())
print("Traducción     :", gen.translate())
print("%GC            : {:.1f}%".format(contenido_gc(str(gen))))

## 7. Leer un archivo FASTA
El formato **FASTA** es el estándar para guardar secuencias. Creamos uno y lo leemos con `SeqIO`.

In [ ]:
fasta = """>gen_demo descripcion de ejemplo
ATGCTAGCTAGCTAGCATGCGCGCTAGCTAGCTAGCTAA
"""
with open("ejemplo.fasta", "w") as f:
    f.write(fasta)

from Bio import SeqIO
for record in SeqIO.parse("ejemplo.fasta", "fasta"):
    print("ID        :", record.id)
    print("Descripción:", record.description)
    print("Longitud  :", len(record.seq))
    print("Proteína  :", record.seq.translate())

## 8. Caso en salud: detectar una mutación
Comparamos una secuencia de referencia con la de un paciente y ubicamos las diferencias (mutaciones).

In [ ]:
referencia = Seq("ATGCGTACGCTTGCA")
paciente   = Seq("ATGCATACGCTTGCA")

mutaciones = []
for i in range(len(referencia)):
    if referencia[i] != paciente[i]:
        mutaciones.append((i, referencia[i], paciente[i]))

if mutaciones:
    for pos, ref, mut in mutaciones:
        print(f"Mutación en posición {pos}: {ref} -> {mut}")
else:
    print("Sin mutaciones: las secuencias son idénticas.")

print("\nProteína referencia:", referencia.translate())
print("Proteína paciente  :", paciente.translate())

---
## 8 bis. Bonus: el ADN en 3D 🧬
Tres formas de visualizar la doble hélice, de más simple a más impresionante.

### a) Hélice 3D con Matplotlib (estática, sin instalar nada)
La generamos con pura matemática: dos hélices desfasadas 180° y los "peldaños" (pares de bases).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

vueltas, n = 4, 300
t = np.linspace(0, vueltas * 2 * np.pi, n)
z = np.linspace(0, 8, n)
r = 1.0
x1, y1 = r * np.cos(t), r * np.sin(t)
x2, y2 = r * np.cos(t + np.pi), r * np.sin(t + np.pi)

fig = plt.figure(figsize=(5, 9))
ax = fig.add_subplot(111, projection="3d")
ax.plot(x1, y1, z, color="#24cbe5", lw=4)   # hebra 1
ax.plot(x2, y2, z, color="#158158", lw=4)   # hebra 2
for i in range(0, n, 12):                     # peldaños = pares de bases
    ax.plot([x1[i], x2[i]], [y1[i], y2[i]], [z[i], z[i]], color="#888", lw=1.5)
ax.set_title("Doble hélice de ADN", color="#08262e", fontweight="bold")
ax.set_axis_off()
plt.show()

### b) Hélice 3D **interactiva** con Plotly
Esta sí la puedes **rotar, hacer zoom y mover con el mouse** dentro de Colab. Los peldaños se colorean según el par de bases.

In [ ]:
import numpy as np
import plotly.graph_objects as go

secuencia = "ATGCTAGCTAGGCCATTAGCGGCTAACGTAGC"
par = {"A": "T", "T": "A", "G": "C", "C": "G"}
color_base = {"A": "#e74c3c", "T": "#3498db", "G": "#f1c40f", "C": "#2ecc71"}

nb = len(secuencia)
t = np.linspace(0, 2.2 * np.pi, nb)
z = np.linspace(0, nb, nb)
r = 1.0
x1, y1 = r * np.cos(t), r * np.sin(t)
x2, y2 = r * np.cos(t + np.pi), r * np.sin(t + np.pi)

fig = go.Figure()
# Las dos hebras del esqueleto
fig.add_trace(go.Scatter3d(x=x1, y=y1, z=z, mode="lines",
                           line=dict(color="#24cbe5", width=10), name="Hebra 5'→3'"))
fig.add_trace(go.Scatter3d(x=x2, y=y2, z=z, mode="lines",
                           line=dict(color="#158158", width=10), name="Hebra 3'→5'"))
# Peldaños = pares de bases
for i, base in enumerate(secuencia):
    fig.add_trace(go.Scatter3d(
        x=[x1[i], x2[i]], y=[y1[i], y2[i]], z=[z[i], z[i]], mode="lines",
        line=dict(color=color_base[base], width=6),
        hovertext=f"{base}–{par[base]}", showlegend=False))
fig.update_layout(
    title="Doble hélice de ADN (interactiva — gírala con el mouse)",
    scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False)),
    paper_bgcolor="#08262e", font=dict(color="white"), height=650)
fig.show()

### c) Estructura **real** de ADN con py3Dmol
Descargamos del Protein Data Bank la estructura `1BNA` (el *dodecámero de Dickerson*, el primer ADN-B resuelto por cristalografía) y la mostramos en 3D, girando. Esto **no es un dibujo**: son las coordenadas atómicas reales.

In [ ]:
!pip install -q py3Dmol

In [ ]:
import py3Dmol

vista = py3Dmol.view(query="pdb:1BNA", width=600, height=500)
vista.setStyle({"cartoon": {"color": "spectrum"}})   # esqueleto en color
vista.addStyle({}, {"stick": {"radius": 0.15}})          # átomos como varillas
vista.setBackgroundColor("0x08262e")
vista.spin(True)                                         # girar automáticamente
vista.zoomTo()
vista.show()

---
## 9. Retos para ti
Resuelve cada uno en las celdas de abajo y sube tu notebook al classroom.

1. **%GC** — Calcula el contenido GC de una secuencia que tú elijas.
2. **Reversa-complementaria** — Usa `reversa_complementaria()` con `"AATTCCGG"`.
3. **Transcribe y traduce** un gen hasta el primer codón STOP.
4. **FASTA** — Crea tu propio archivo FASTA y léelo con Biopython, imprime su longitud.
5. **3D** — Cambia la `secuencia` de la hélice de Plotly por una tuya y vuelve a graficarla.
6. **Bonus** — Compara dos secuencias y reporta todas las mutaciones.

In [ ]:
# Reto 1: tu código aquí


In [ ]:
# Reto 2: tu código aquí


In [ ]:
# Reto 3: tu código aquí


In [ ]:
# Reto 4: tu código aquí


In [ ]:
# Reto 5: tu código aquí (hélice 3D con tu secuencia)


In [ ]:
# Reto 6 (bonus): tu código aquí
